In [ ]:
import nflreadpy as nfl

ff_rankings = nfl.load_ff_rankings()
print(ff_rankings.shape)
print(ff_rankings.columns)
print(ff_rankings.head())



In [ ]:
print(ff_rankings["ecr_type"].value_counts())
print(ff_rankings["page_type"].value_counts())
#print(ff_rankings["ecr_type"].unique())


In [ ]:
players = ff_rankings.filter(
    ff_rankings["pos"].is_in(["QB", "RB", "WR", "TE"])
)

print(players.shape)

print(
    players.select([
        "player",
        "pos",
        "team",
        "ecr",
        "best",
        "worst",
        "bye",
        "ecr_type"
    ]).head(30)
)

In [ ]:
redraft = ff_rankings.filter(
    ff_rankings["page_type"] == "redraft-overall"
)

print(redraft.shape)

print(
    redraft.select([
        "player",
        "pos",
        "team",
        "ecr",
        "best",
        "worst",
        "bye"
    ]).sort("ecr").head(50)
)

In [ ]:
draft_board = redraft.select([
    "player",
    "pos",
    "team",
    "ecr",
    "best",
    "worst",
    "bye"
]).sort("ecr")

draft_board.head(100)

In [ ]:
import polars as pl

draft_board = draft_board.with_columns(
    (pl.col("worst") - pl.col("best")).alias("expert_range")
)

draft_board.select([
    "player",
    "pos",
    "ecr",
    "best",
    "worst",
    "expert_range"
]).sort("ecr").head(50)



In [ ]:
redraft.select([
    "player", "pos", "team", "ecr", "best", "worst", "bye"
]).sort("ecr").head(30)

In [ ]:
import polars as pl

draft_board = redraft.select([
    "player",
    "pos",
    "team",
    "ecr",
    "best",
    "worst",
    "bye"
]).with_columns(
    (pl.col("worst") - pl.col("best")).alias("expert_range")
).sort("ecr")

print(
    draft_board.select([
        "player",
        "pos",
        "ecr",
        "best",
        "worst",
        "expert_range"
    ]).head(50)
)

In [ ]:
draft_board = draft_board.with_columns(
    (pl.col("expert_range") / pl.col("ecr")).alias("uncertainty")
)

draft_board.select([
    "player",
    "pos",
    "ecr",
    "best",
    "worst",
    "expert_range",
    "uncertainty"
]).sort("ecr").head(30)



In [ ]:
draft_board = draft_board.with_columns(
    (
        100 / pl.col("ecr")
        - 2 * pl.col("uncertainty")
    ).alias("draft_score")
)

draft_board.sort("draft_score", descending=True).head(50)

In [ ]:
position_rank = (
    draft_board
    .with_columns(
        pl.col("ecr")
        .rank()
        .over("pos")
        .alias("position_rank")
    )
)

In [ ]:
draft_board = (
    redraft
    .select([
        "player",
        "pos",
        "team",
        "ecr",
        "best",
        "worst",
        "bye"
    ])
    .sort("ecr")
)

print(draft_board.head(50))

In [ ]:
draft_board = draft_board.with_columns(
    (pl.col("worst") - pl.col("best")).alias("expert_range")
)

In [ ]:
draft_board = draft_board.with_columns(
    (1 / (1 + pl.col("expert_range"))).alias("confidence")
)

In [ ]:
draft_board = draft_board.with_columns(
    (
        100 - pl.col("ecr")
    ).alias("draft_score")
)

In [ ]:
draft_board = draft_board.sort(
    "draft_score",
    descending=True
)

print(
    draft_board.select([
        "player",
        "pos",
        "team",
        "ecr",
        "best",
        "worst",
        "expert_range",
        "confidence"
    ]).head(50)
)

In [61]:
import polars as pl

def show_position(position, n=15):
    position = position.upper()

    filtered = (
        draft_board
        .filter(pl.col("pos") == position)
        .sort("ecr")
        .select([
            "player",
            "pos",
            "team",
            "ecr",
            "best",
            "worst",
            "expert_range",
            "confidence"
        ])
        .head(n)
    )

    print(f"\n===== TOP {n} {position}s =====")
    
    return filtered

In [ ]:
show_position("RB")

In [63]:
show_position("QB")


===== TOP 15 QBs =====


player,pos,team,ecr,best,worst,expert_range,confidence
str,str,str,f64,i64,i64,i64,f64
"""Josh Allen""","""QB""","""BUF""",25.9,21,37,16,0.058824
"""Lamar Jackson""","""QB""","""BAL""",33.52,25,70,45,0.021739
"""Drake Maye""","""QB""","""NE""",38.28,26,119,93,0.010638
"""Joe Burrow""","""QB""","""CIN""",46.79,27,101,74,0.013333
"""Jayden Daniels""","""QB""","""WAS""",54.35,27,108,81,0.012195
…,…,…,…,…,…,…,…
"""Brock Purdy""","""QB""","""SF""",96.48,63,130,67,0.014706
"""Jaxson Dart""","""QB""","""NYG""",97.98,59,129,70,0.014085
"""Patrick Mahomes II""","""QB""","""KC""",100.36,70,138,68,0.014493


In [64]:
show_position("DST")


===== TOP 15 DSTs =====


player,pos,team,ecr,best,worst,expert_range,confidence
str,str,str,f64,i64,i64,i64,f64
"""Houston Texans""","""DST""","""HOU""",153.71,143,176,33,0.029412
"""Denver Broncos""","""DST""","""DEN""",163.01,149,210,61,0.016129
"""Los Angeles Rams""","""DST""","""LAR""",167.31,148,186,38,0.025641
"""Seattle Seahawks""","""DST""","""SEA""",167.56,151,194,43,0.022727
"""Philadelphia Eagles""","""DST""","""PHI""",174.91,156,235,79,0.0125
…,…,…,…,…,…,…,…
"""Baltimore Ravens""","""DST""","""BAL""",199.59,171,356,185,0.005376
"""Green Bay Packers""","""DST""","""GB""",203.24,184,280,96,0.010309
"""Kansas City Chiefs""","""DST""","""KC""",210.3,151,280,129,0.007692
